In [1]:
import os
cwd = os.getcwd()
print("当前目录:", cwd)
path = os.path.join(cwd, "my_chroma_db")
print("可写?", os.access(cwd, os.W_OK))

当前目录: /Users/ludan/project/chanon-data-lab/notebookes
可写? True


In [2]:
import os
import unstructured
# 使用 Hugging Face 国内镜像，避免 Read timed out（必须在 import 相关库之前设置）
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    Docx2txtLoader,
    CSVLoader,
    UnstructuredMarkdownLoader,
)
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_docs(path: str):
    """按扩展名选加载器，返回 Document 列表。支持: .pdf / .txt / .docx / .csv / .md"""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        return PyPDFLoader(path).load_and_split()
    if ext == ".txt":
        return TextLoader(path, encoding="utf-8", autodetect_encoding=True).load()
    if ext == ".docx":
        return Docx2txtLoader(path).load()
    if ext == ".md":
        return UnstructuredMarkdownLoader(path).load()
    if ext == ".csv":
        return CSVLoader(path, encoding="utf-8").load()
    raise ValueError(f"不支持的后缀: {ext}，可用: .pdf .txt .docx .md .csv")

SUPPORTED_EXT = (".pdf", ".txt", ".docx", ".md", ".csv")

def list_supported_files(folder: str, recursive: bool = True):
    """收集文件夹下所有支持格式的文件路径（默认含子文件夹）。"""
    paths = []
    folder = os.path.abspath(os.path.expanduser(folder))
    if not os.path.isdir(folder):
        return paths
    for root, _, files in os.walk(folder):
        for name in files:
            if os.path.splitext(name)[1].lower() in SUPPORTED_EXT:
                paths.append(os.path.join(root, name))
        if not recursive:
            break
    return sorted(paths)

# 1. 加载：可填「文件夹」（会加载该目录及子目录下所有 .pdf/.txt/.docx/.md/.csv），或单独文件路径
folder_paths = [
    "/Users/ludan/Documents/Obsidian Vault/清明节被放出来的鬼",
]
file_paths = [
    "/Users/ludan/Downloads/大模型导论.pdf",
    "/Users/ludan/Downloads/深度学习.pdf",
]
all_paths = []
for folder in folder_paths:
    all_paths.extend(list_supported_files(folder, recursive=True))
all_paths.extend(file_paths)
all_paths = list(dict.fromkeys(all_paths))  # 去重保持顺序

pages = []
for path in all_paths:
    pages.extend(load_docs(path))
print(f"共加载 {len(pages)} 个文档片段，来自 {len(all_paths)} 个文件")

# 1.5 切分为较短片段再入库，避免「整页」过长导致超出模型上下文或难以定位答案
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80, length_function=len)
chunks = splitter.split_documents(pages)
print(f"切分后共 {len(chunks)} 个片段（每段约 500 字，便于检索且不超长）")

# 2. 存入 Chroma（内存模式，不写磁盘，避免沙箱/只读导致的 "readonly database"）
# 每次运行本 cell 会重新建库；若需持久化且环境可写，再改回 persist_directory=某路径
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"),
)
print("Chroma 已就绪（内存模式）")

/Users/ludan/project/chanon-data-lab/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


共加载 1086 个文档片段，来自 11 个文件
切分后共 2759 个片段（每段约 500 字，便于检索且不超长）


/var/folders/h3/7zc3322s3y1bwclms8c2rjwc0000gn/T/ipykernel_404/3131852052.py:77: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"),


Chroma 已就绪（内存模式）


In [3]:
def dedupe_docs_by_page(docs):
    """按 (source, page) 去重，避免同一页出现多份（常因多次写入同一 persist_directory 导致）"""
    seen = set()
    out = []
    for doc in docs:
        meta = doc.metadata
        key = (meta.get("source"), meta.get("page"), meta.get("page_label"))
        if key in seen:
            continue
        seen.add(key)
        out.append(doc)
    return out

def format_docs(docs, sep="─" * 60, dedupe=True):
    """把 LangChain Document 列表按「来源 / 页码 + 正文」格式化打印"""
    if dedupe:
        docs = dedupe_docs_by_page(docs)
    for i, doc in enumerate(docs, 1):
        meta = doc.metadata
        source = meta.get("source", "")
        page = meta.get("page") or meta.get("page_label", "?")
        print(f"{sep}")
        print(f"[{i}] 来源: {source}  页码: {page}")
        print(f"{sep}")
        print(doc.page_content)
        print()


# # 3. 检索
# docs = vectorstore.similarity_search("关于 Transformer 的细节")
# # 格式化输出检索结果（自动按来源+页码去重）
# format_docs(docs)

In [4]:
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

LM_STUDIO_BASE = "http://127.0.0.1:1234/v1"
llm = ChatOpenAI(
    base_url=LM_STUDIO_BASE,
    api_key="lm-studio",
    temperature=0,
    model="local",
)

# 优先根据参考资料回答；只有完全无关时才说「无法得出答案」（避免因上下文过长/模型保守而总说无法得出）
QA_PROMPT = PromptTemplate(
    template="""请根据以下「参考资料」回答问题。若资料中有与问题相关的内容，请据此简洁回答；若完全没有相关内容，再回答「根据提供的资料无法得出答案」。

参考资料：
{context}

问题：{question}

请根据上述参考资料回答：""",
    input_variables=["context", "question"],
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
pdf_qa = RetrievalQA.from_chain_type(
    llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": QA_PROMPT},
    return_source_documents=True,
)

In [5]:

query = "什么是transformer？"
result = pdf_qa.invoke({"query": query})
print("Answer:", result["result"])
# 展示本次回答用到的知识库片段（证明用上了 PDF）
print("\n--- 本次检索到的参考资料（来自 PDF）---")
format_docs(result["source_documents"])

Answer: Transformer 是一种自注意力网络模型，用于处理序列（如文本、图像）数据。它由四个主要模块组成：输入模块、编码器（Encoder）、解码器（Decoder）以及输出模块。通过位置编码和多‑head 自注意力子层，Transformer能够捕捉词/符号之间的语义与位置信息，并将这些信息经过残差连接与归一化操作送到下层，最终产生概率或数值结果。其结构使得 Transformer 在 NLP、机器翻译、文本生成以及视觉任务（如 Visual‑Transformer）取得优异性能。

--- 本次检索到的参考资料（来自 PDF）---
────────────────────────────────────────────────────────────
[1] 来源: /Users/ludan/Downloads/大模型导论.pdf  页码: 100
────────────────────────────────────────────────────────────
86 第 3 章 Transformer
(l) 简述自注意力机制的特点。
(2) 位置编码的作用是什么？
3.7 课后习题
(3) transformers 库主要提供哪儿类模型？
(4) 按照 3.5 节提供的案例，自行在本地进行相关实践，体会 transformers 库的用法 。

────────────────────────────────────────────────────────────
[2] 来源: /Users/ludan/Downloads/大模型导论.pdf  页码: 85
────────────────────────────────────────────────────────────
3.2 Transformer 简介 71 
通过引入位置编码， Transformer 模型能够有效地处理序列数据，同时捕捉 Token 之间的
语义和位置关系。这对千 NLP 等领域的任务来说至关重要，也是 Transformer 模型能够取得
优异性能的重要原因之 一。
3.2.2 整体结构
Transformer 的整体结构可分为输入模块、编码器模块、解码器模块和输出模块，如图 3-8
所示 。 seq2seq 架构的 Transformer 模型包含